# TODO: You must set checkpoint_directory to where you want to save and load the model

In [1]:
checkpoint_base_directory ="/Users/jameswalker/Programming/Checkpoints3/MappoCheckpoint"
checkpoint_number = 0

In [2]:
import ray 

ray.shutdown()
ray.init() 

2025-04-27 12:03:40,600	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [3]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.marl_envs.roundabout_rllib_delegator_env import RoundaboutRLLibDelegatorEnv 
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec

# Not sure if sgd_minibatch_size is being handled properly, but train_batch_size and worker_num are for sure
# worker_num = 1
# train_batch_size = int(1024)
# sgd_minibatch_size = max(256, int(train_batch_size/10))

config = ( PPOConfig()
    .environment(
        RoundaboutRLLibDelegatorEnv,
        disable_env_checking=True
        )
    .env_runners(
        num_env_runners=3, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0
        )
    .multi_agent(
        policies={"agent0", "agent1"},
        policy_mapping_fn=lambda agent_id, episode, **kw: agent_id,
    )
    .framework('torch')
    # The below .learners comes from the migration guide when they talk about no GPU but multi-CPU
    .learners(
        num_learners=2,
        num_cpus_per_learner=2,
        # num_gpus_per_learner=0,
    )
    .training(
        # train_batch_size=train_batch_size, # Deprecated
        train_batch_size_per_learner=int(2048), # Replacement for train_batch_size
        # sgd_minibatch_size=sgd_minibatch_size, # This got deprecated
        minibatch_size=int(256), # This might be the replacement
        lr=1e-4,
        entropy_coeff=0.001,
    )    
    .rl_module(
        # Use a non-default 32,32-stack with ReLU activations.
        # This is for PPO since it uses multi-layer perceptrons (MLP)
        model_config=DefaultModelConfig(
            fcnet_hiddens=[32, 32],
            fcnet_activation="relu",
        ),
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
                "agent0": RLModuleSpec(),
                "agent1": RLModuleSpec(),
            }
        ),
    )
    # The below, commented .training illustrates grid search for learning rate,
    # comes from https://github.com/ray-project/ray/blob/master/rllib/examples/gpus/fractional_gpus_per_learner.py
    # .training(
    #     lr=tune.grid_search([0.005, 0.003, 0.001, 0.0001])
    # )
    # .evaluation(
    #     # TODO evaluation_interval?
    #     # evaluation_interval=1,
    #     evaluation_num_env_runners=1, 
    #     # evaluation_duration=10000,
    #     # WARNING algorithm_config.py:4593 -- You have specified 1 evaluation workers, but your `evaluation_interval` is 0 or None! Therefore, evaluation doesn't occur automatically with each call to `Algorithm.train()`. Instead, you have to call `Algorithm.evaluate()` manually in order to trigger an evaluation run.
    # )
    # .resources is only for setting num_cpus_for_main_process, I don't think we need to mess with it
    # .resources(
    #     num_gpus=int(0),  # This is deprecated
    #     num_cpus_for_main_process=int(1), # Defaults to 1
    # )  
    .debugging(log_level='INFO') # INFO, DEBUG, ERROR, WARN
)



In [4]:
# config.build() is deprecated, use build_algo() instead
algo = config.build_algo()
# print(f"Type of algo: {type(algo)}")


2025-04-27 12:03:48,799	WARNING algorithm_config.py:4704 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this wa

(_WrappedExecutable pid=46733) Setting up process group for: env:// [rank=0, world_size=2]
(_WrappedExecutable pid=46733) 2025-04-27 12:03:58,537	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


# TRAIN THE ALGORITHM
100 training iterations took about 2 minutes on my Mac with M1 chip
If setting it up on a server, just try 100 iterations and see if you can save and load the Algorithm instance. After that try 10,000 iterations

In [5]:
# from collections import deque
# SMOOTHING_WINDOW = 10  
# returns_window = deque(maxlen=SMOOTHING_WINDOW)

iteration_num = 1_000
ENV_RUNNER_RESULTS = "env_runners"
EPISODE_RETURN_MEAN = "episode_return_mean"
EVALUATION_RESULTS = "evaluation"
for iteration in tqdm(range(0, iteration_num)):
    results = algo.train()
    # # train_R = results["env_runners"].get("episode_return_mean", float("nan"))
    # # returns_window.append(train_R)

    # # # compute the smoothed average
    # # smooth_R = sum(returns_window) / len(returns_window)

    # # print(f"it={iteration:4d}   R_train={train_R:6.2f}   R_smooth={smooth_R:6.2f}", end="")

    # # # if you’ve enabled built-in evaluation in your config:
    # # if "evaluation" in results and "env_runners" in results["evaluation"]:
    # #     eval_R = results["evaluation"]["env_runners"]["episode_return_mean"]
    # #     print(f"   R_eval={eval_R:6.2f}", end="")

    # print()
    if iteration % 10 == 0:
        # Print metrics every 10 iterations
        # Save checkpoint every 50 iterations
        if iteration % 50 == 0:
            # Save Algorithm checkpoint.
            checkpoint_directory = checkpoint_base_directory + str(checkpoint_number)
            algo.save_to_path(checkpoint_directory)
            checkpoint_number = checkpoint_number + 1
            print(f"Saved to {checkpoint_directory}")

        if ENV_RUNNER_RESULTS in results:
            mean_return = results["env_runners"].get("episode_return_mean", float("nan"))
            print(f"iter={iteration} R={mean_return}", end="")
            if (EVALUATION_RESULTS in results and ENV_RUNNER_RESULTS in results[EVALUATION_RESULTS]):
                Reval = results[EVALUATION_RESULTS][ENV_RUNNER_RESULTS][EPISODE_RETURN_MEAN]
                print(f" R(eval)={Reval}", end="")
            print()
    # For seeing all the metrics we can use in the result dict():
    # for key in result['env_runners'].keys():
    #     print(f"Key {key}: {result['env_runners'][key]}")
    # # TODO Are these good metrics below? Not sure exactly what either means
    # if iteration % (iteration_num/10) == 0:
        # print(f"Iteration {iteration} Episode Length Mean: {result['env_runners']['episode_len_mean']}")
        # print(f"Iteration {iteration} Agent Episode Returns Mean: {result['env_runners']['agent_episode_returns_mean']}")


  0%|          | 0/1000 [00:00<?, ?it/s]

(MultiAgentEnvRunner pid=46738) [INFO] Assets version: 0.4.3
(_WrappedExecutable pid=46751) 2025-04-27 12:03:58,537	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
(MultiAgentEnvRunner pid=46738) [INFO] Known Pipes: CocoaGraphicsPipe
(MultiAgentEnvRunner pid=46738) [INFO] Start Scenario Index: 0, Num Scenarios : 1
  0%|          | 1/1000 [00:10<2:54:05, 10.46s/it]

Saved to /Users/jameswalker/Programming/Checkpoints3/MappoCheckpoint0
iter=0 R=-4531.598371766455


  1%|          | 11/1000 [01:26<2:02:57,  7.46s/it]

iter=10 R=-8057.856871054787


...

iter=140 R=-17321.611780742758


 15%|█▌        | 151/1000 [17:42<1:37:18,  6.88s/it]

Saved to /Users/jameswalker/Programming/Checkpoints3/MappoCheckpoint3
iter=150 R=-16665.817205788313


 16%|█▌        | 155/1000 [18:10<1:38:01,  6.96s/it]2025-04-27 12:52:20,931	ERROR actor_manager.py:873 -- Ray error (ray::_WrappedExecutable.update() (pid=46733, ip=127.0.0.1, actor_id=be194bdd654d45d5c13fd15801000000, repr=<ray.train._internal.worker_group._WrappedExecutable object at 0x17789d220>)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/learner.py", line 1121, in update
    fwd_out, loss_per_module, tensor_metrics = self._update(
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/torch/torch_learner.py", line 513, in _update
    return self._possibly_compiled_update(batch)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/torch/torch_learner.py", line 156, in _uncompiled_update
    fwd_out = self.module.forward_train(batch)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/rl_module/rl_module.py", line 630, in forward_train
    r

RayTaskError(RuntimeError): [36mray::_WrappedExecutable.update()[39m (pid=46733, ip=127.0.0.1, actor_id=be194bdd654d45d5c13fd15801000000, repr=<ray.train._internal.worker_group._WrappedExecutable object at 0x17789d220>)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/learner.py", line 1121, in update
    fwd_out, loss_per_module, tensor_metrics = self._update(
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/torch/torch_learner.py", line 513, in _update
    return self._possibly_compiled_update(batch)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/learner/torch/torch_learner.py", line 156, in _uncompiled_update
    fwd_out = self.module.forward_train(batch)
  File "/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/core/rl_module/rl_module.py", line 630, in forward_train
    return self._forward_train(batch, **kwargs)
  File "/Users/jameswalker/miniconda

# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [6]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [3]:
env.close()